# Файнтюн чемпиона Qwen3-32B QLoRA в Google Colab

Прогон **одной** лучшей модели по итогам 3 раундов на сервере MEPhI (`best_finetune_configs/qwen3_32b_qlora.json`) с двумя свежими исправлениями:
1. **Class weights в loss** (`use_class_weights=true`) — главный путь к +3-7pt на macro_f1
2. **Train/val split 90/10 stratified** (`val_split=0.10`) — устраняет selection bias (test больше не виден при early stopping)

**Что нужно:**
- Colab Pro / Pro+ с **A100-40GB** или хотя бы **L4-24GB** (Qwen3-32B QLoRA в 4bit ≈ 18 GB + активации)
- Папка на Google Drive `MyDrive/VKR/` с подпапкой `Data/` (CSV-данные)
- GitHub-токен если репо приватный (раскомментируй ячейку с PAT)

**Что сохранится на Drive:**
- `MyDrive/VKR/finetune_runs/<timestamp>/adapter/` — веса адаптера, токенизатор, метаданные
- `MyDrive/VKR/finetune_runs/<timestamp>/results/` — finetune_results.csv + preds_*.csv
- `MyDrive/VKR/finetune_runs/<timestamp>/logs/` — stdout лог обучения
- `MyDrive/VKR/finetune_runs/<timestamp>/config_used.json` — копия использованного конфига

## 1. Монтируем Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path
from datetime import datetime

DRIVE_ROOT = Path('/content/drive/MyDrive/VKR')
DRIVE_DATA = DRIVE_ROOT / 'Data'
RUN_TS = datetime.now().strftime('%Y%m%d_%H%M%S')
DRIVE_RUN = DRIVE_ROOT / 'finetune_runs' / RUN_TS
DRIVE_RUN.mkdir(parents=True, exist_ok=True)

print(f'Drive root      : {DRIVE_ROOT} (exists={DRIVE_ROOT.exists()})')
print(f'Drive Data      : {DRIVE_DATA} (exists={DRIVE_DATA.exists()})')
print(f'Run output      : {DRIVE_RUN}')

assert DRIVE_DATA.exists(), (
    f'Папка с данными {DRIVE_DATA} не найдена. '
    'Положи на Drive: data_after_stage3.csv, data_test.csv, train_after_eda.csv.'
)

## 2. Клонируем / обновляем код из GitHub

Код кладётся в `/content/VKR` (локальный SSD Colab — быстрее чем Drive). Если репо публичный — `git clone` сработает сразу. Если приватный — раскомментируй ячейку с PAT (Personal Access Token) и подставь свой токен.

In [ ]:
# === Если репо ПРИВАТНЫЙ — раскомментируй и подставь PAT ===
# Создать токен: https://github.com/settings/tokens (scope=repo)
# GITHUB_PAT = 'ghp_xxxxxxxxxxxxxxxxxxxx'
# REPO_URL = f'https://{GITHUB_PAT}@github.com/KVTur23/Mifi_VKR.git'

# === Публичный репо — оставь так ===
REPO_URL = 'https://github.com/KVTur23/Mifi_VKR.git'
REPO_BRANCH = 'fine-tune'  # ветка с актуальным кодом (class_weights + val_split)
REPO_DIR = Path('/content/VKR')

In [ ]:
import subprocess

if (REPO_DIR / '.git').exists():
    print(f'Репо уже клонирован — git pull')
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', 'origin', REPO_BRANCH], check=True)
else:
    print(f'Клонирую {REPO_URL} в {REPO_DIR}')
    subprocess.run(['git', 'clone', '-b', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

# Текущий коммит — для воспроизводимости
commit = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'log', '-1', '--oneline'],
    capture_output=True, text=True, check=True,
).stdout.strip()
print(f'\nТекущий коммит: {commit}')

## 3. Подцепляем данные с Drive

Symlink `Data/` из Drive в репо — никаких копирований, файлы остаются на Drive. Чекпоинты адаптера будут писаться в `/content/VKR/Data/finetune_checkpoints/` (локальный SSD), потом мы их сами скопируем на Drive.

In [ ]:
PROJECT_CODE = REPO_DIR / 'code'  # код лежит в подпапке code/ репозитория
LOCAL_DATA = PROJECT_CODE / 'Data'

# Если в репо есть пустая Data/ — переименовываем, чтобы не конфликтовала
if LOCAL_DATA.exists() and not LOCAL_DATA.is_symlink():
    LOCAL_DATA.rename(LOCAL_DATA.parent / 'Data_repo_orig')

# Symlink CSV-файлов с Drive (чекпоинты будем хранить локально, тяжёлые)
LOCAL_DATA.mkdir(exist_ok=True)
for csv in DRIVE_DATA.glob('*.csv'):
    target = LOCAL_DATA / csv.name
    if not target.exists():
        target.symlink_to(csv)

(LOCAL_DATA / 'finetune_checkpoints').mkdir(exist_ok=True)
(PROJECT_CODE / 'results').mkdir(exist_ok=True)
(PROJECT_CODE / 'logs').mkdir(exist_ok=True)

print(f'Data в репо:')
for f in sorted(LOCAL_DATA.iterdir()):
    if f.is_symlink():
        print(f'  {f.name} -> {f.readlink()}')
    else:
        print(f'  {f.name}/')

## 4. Устанавливаем зависимости

In [ ]:
# Базовые пакеты для FT (без vLLM — он не нужен здесь и долго ставится)
!pip install -q transformers peft bitsandbytes accelerate datasets sentencepiece scikit-learn pandas filelock 2>&1 | tail -5

## 5. Проверка GPU

In [ ]:
import torch
import sys

if str(PROJECT_CODE) not in sys.path:
    sys.path.insert(0, str(PROJECT_CODE))

if not torch.cuda.is_available():
    raise RuntimeError('CUDA недоступна. Runtime → Change runtime type → A100/L4 GPU')

name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {name}  ({vram_gb:.1f} GB)')

# Подбираем GPU-профиль по VRAM
if vram_gb >= 75:
    GPU_PROFILE = 'A100_80'
elif vram_gb >= 38:
    GPU_PROFILE = 'A100_40'
elif vram_gb >= 22:
    GPU_PROFILE = 'L4'
    print('⚠ L4 24GB — Qwen3-32B QLoRA впритык, может быть OOM. Лучше A100.')
else:
    raise RuntimeError(f'GPU слишком маленькая ({vram_gb:.1f} GB) для Qwen3-32B QLoRA. Нужно ≥24GB.')

print(f'Профиль для pipeline_config: {GPU_PROFILE}')

## 6. Используем `best_finetune_configs/qwen3_32b_qlora.json`

Берём чемпиона из R1 (r=16, dropout=0.05, lr=2e-4, epochs=5) и накладываем поверх:
- `use_class_weights: true` — главный апгрейд
- `val_split: 0.10` — устранение selection bias

Конфиг копируется в `/content/VKR/code/config_models/finetune_configs/qwen3_32b_qlora.json` (orchestrator берёт оттуда).

In [ ]:
import json, shutil

# best-конфиг лежит в репо (мы его сохранили в experiment/, но в repo код берёт из config_models/)
# Просто читаем актуальный конфиг из код-папки и проверяем что флаги выставлены
config_path = PROJECT_CODE / 'config_models' / 'finetune_configs' / 'qwen3_32b_qlora.json'

with open(config_path) as f:
    cfg = json.load(f)

# Гарантируем флаги
cfg['use_class_weights'] = True
cfg['val_split'] = 0.10

with open(config_path, 'w') as f:
    json.dump(cfg, f, ensure_ascii=False, indent=2)

# Копия в Drive для воспроизводимости
shutil.copy(config_path, DRIVE_RUN / 'config_used.json')

print('=== config_used ===')
print(json.dumps(cfg, ensure_ascii=False, indent=2))

## 7. Запуск (1 модель, ~2-3 часа на A100-40)

Логи дублируются в файл, который потом сохраним на Drive.

In [ ]:
%cd {PROJECT_CODE}

import logging, sys
log_path = PROJECT_CODE / 'logs' / f'ft_qlora_qwen3_32b_{RUN_TS}.log'

# Tee stdout/stderr в файл (чтобы потом сохранить на Drive)
class Tee:
    def __init__(self, *files): self.files = files
    def write(self, s):
        for f in self.files: f.write(s); f.flush()
    def flush(self):
        for f in self.files: f.flush()

_log_file = open(log_path, 'w', encoding='utf-8')
sys.stdout = Tee(sys.__stdout__, _log_file)
sys.stderr = Tee(sys.__stderr__, _log_file)

print(f'=== START | {datetime.now().isoformat()} | log → {log_path} ===\n')

In [ ]:
from src.finetune.orchestrator import run_finetune

df = run_finetune(
    methods=['qlora_qwen3_32b'],  # один чемпион
    gpu=GPU_PROFILE,
    force=True,                    # на всякий — переписать прежний результат если есть
)
df

In [ ]:
# Восстановить stdout/stderr и закрыть файл-лог
sys.stdout = sys.__stdout__
sys.stderr = sys.__stderr__
_log_file.close()
print(f'Log сохранён локально: {log_path} ({log_path.stat().st_size / 1024:.1f} KB)')

## 8. Сохраняем всё полезное на Drive

**Адаптер** (200-500 МБ — терпимо), **результаты CSV**, **лог обучения**, **id2label / class_groups / metadata**.

In [ ]:
import shutil

# 1. Адаптер целиком (включая токенизатор, id2label.json, metadata.json, adapter_model.safetensors)
src_adapter = PROJECT_CODE / 'Data' / 'finetune_checkpoints' / 'qlora_qwen3_32b'
dst_adapter = DRIVE_RUN / 'adapter'

if src_adapter.exists():
    print(f'Копирую адаптер → {dst_adapter} (это может занять 1-3 минуты, файлы крупные)...')
    shutil.copytree(src_adapter, dst_adapter, dirs_exist_ok=True)
    sz = sum(f.stat().st_size for f in dst_adapter.rglob('*') if f.is_file()) / 1e6
    print(f'✓ Адаптер: {sz:.1f} МБ')
else:
    print(f'⚠ Адаптер не найден: {src_adapter}')

# 2. Результаты
src_results = PROJECT_CODE / 'results'
dst_results = DRIVE_RUN / 'results'
shutil.copytree(src_results, dst_results, dirs_exist_ok=True)
print(f'✓ Results → {dst_results}')

# 3. Лог обучения
dst_logs = DRIVE_RUN / 'logs'
dst_logs.mkdir(exist_ok=True)
if log_path.exists():
    shutil.copy(log_path, dst_logs / log_path.name)
    print(f'✓ Log → {dst_logs / log_path.name}')

# 4. Manifest — что в этом ране
manifest = {
    'timestamp': RUN_TS,
    'gpu': name,
    'gpu_profile': GPU_PROFILE,
    'commit': commit,
    'method': 'qlora_qwen3_32b',
    'config': cfg,
    'metrics': df.to_dict('records') if not df.empty else [],
}
with open(DRIVE_RUN / 'manifest.json', 'w') as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)
print(f'\n✓ Manifest → {DRIVE_RUN / "manifest.json"}')

## 9. Финальные метрики

Если получилось **macro_f1 > 0.55** или **balanced_accuracy > 0.58** — class_weights отработал, можно катать остальные модели тем же ноутбуком, заменив в ячейке 7 method на `lora_qwen3_14b`, `qlora_tpro_it_21` и т.д.

In [ ]:
import pandas as pd

results_csv = PROJECT_CODE / 'results' / 'finetune_results.csv'
if results_csv.exists():
    df_full = pd.read_csv(results_csv)
    print('=== finetune_results.csv ===\n')
    print(df_full.to_string(index=False))

    # Сравнение с прежними цифрами (R1 чемпион — без class_weights и без val_split)
    print('\n=== Сравнение с baseline (R1 без фиксов) ===')
    print('R1 baseline qlora_qwen3_32b:  bal_acc=0.5615, macro_f1=0.5228, f1_C=0.625')
    if 'qlora_qwen3_32b' in df_full['run_key'].values:
        row = df_full[df_full['run_key'] == 'qlora_qwen3_32b'].iloc[0]
        print(f'Текущий прогон:               bal_acc={row["balanced_accuracy"]:.4f}, '
              f'macro_f1={row["macro_f1"]:.4f}, f1_C={row["f1_group_C"]:.4f}')
        delta_ba = row['balanced_accuracy'] - 0.5615
        delta_f1 = row['macro_f1'] - 0.5228
        print(f'\nΔ bal_acc: {"+" if delta_ba >= 0 else ""}{delta_ba:.4f}')
        print(f'Δ macro_f1: {"+" if delta_f1 >= 0 else ""}{delta_f1:.4f}')
        print('\nNB: baseline имел selection bias (~+1-2pt), текущая цифра честная.')
else:
    print('⚠ finetune_results.csv не найден')